# 09. InfraScan Crack Segmentation Training (U-Net & U-Net++) - Kaggle Edition

This notebook trains crack segmentation models on the CRACK500 dataset using **segmentation-models-pytorch**.

### Kaggle Manual Setup Steps:
1. **Upload Dataset**:
   - Create a private Kaggle dataset containing `crack500_seg.zip` and `ml_code_segmentation.zip` (from `data/processed/colab_zips/`).
2. **Attach Input**:
   - In the right-hand panel, click **+ Add Input** and attach your dataset.
3. **Accelerator & Internet**:
   - Set **Accelerator** to **GPU T4 x2** or **P100**.
   - Toggle **Internet** to **On** in Notebook Settings (for `pip install segmentation-models-pytorch` and ImageNet pretrained backbone weights).
4. **Save Version**:
   - When training completes, click **Save Version** (e.g. *Save & Run All* or *Quick Save*) to persist all checkpoints and download `/kaggle/working/segmentation_results.zip`.

> [!WARNING]
> **SCIENTIFIC INTEGRITY GUARD**: The test split must NEVER be evaluated during training or model selection. All model comparison and threshold tuning are conducted strictly on the **VALIDATION split**. Test split evaluation is executed only after model selection in Notebook 10.

### 1. Check and Install Dependencies

Use an existing `segmentation-models-pytorch` install when available. Internet is needed only if pip must install the package or download uncached ImageNet encoder weights.

In [ ]:
import socket
import subprocess
import sys

def check_internet(host="8.8.8.8", port=53, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except (socket.error, OSError):
        return False

# Step 1: Check if segmentation-models-pytorch is already installed
try:
    import segmentation_models_pytorch as smp
    print(f"[OK] segmentation-models-pytorch is already installed (version: {smp.__version__}).")
except ModuleNotFoundError:
    # Step 2: Check internet connection before attempting pip install
    if not check_internet():
        print("=" * 75)
        print("[!] ACTION REQUIRED: INTERNET ACCESS IS OFF IN THIS KAGGLE SESSION")
        print("=" * 75)
        print("Kaggle disables Internet by default when creating or copying notebooks.")
        print("Internet access is required to install dependencies and download ImageNet weights.")
        print("")
        print(">>> HOW TO TURN INTERNET ON (Takes 5 seconds):")
        print("  1. Look at the RIGHT-HAND panel of your Kaggle notebook.")
        print("  2. Scroll down to the 'Notebook settings' section.")
        print("  3. Find 'Internet' and click the toggle switch to turn it ON.")
        print("  4. Re-run this cell!")
        print("=" * 75)
        raise ConnectionError("Internet is turned OFF. Please toggle Internet to ON in the right sidebar Settings panel.")

    print("[INFO] Internet verified. Installing segmentation-models-pytorch via pip...")
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "segmentation-models-pytorch"],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        print("[!] Pip install output:")
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f"Pip install failed (exit status {res.returncode}). See output above.")
    
    import segmentation_models_pytorch as smp
    print(f"[OK] segmentation-models-pytorch successfully installed (version: {smp.__version__}).")


### 2. Verify GPU Accelerator and Free Disk Space

Audit hardware acceleration (`device=0`) and inspect available disk space in `/kaggle/working`.

In [ ]:
import shutil
import torch

if torch.cuda.is_available():
    torch.cuda.set_device(0)

print("=" * 65)
print("HARDWARE & STORAGE AUDIT (device=0)")
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name (0)    : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Capacity   : {vram_gb:.2f} GB")
else:
    print("[!] WARNING: GPU accelerator not detected. Please select GPU in Settings.")

working_dir = "/kaggle/working"
total_b, used_b, free_b = shutil.disk_usage(working_dir)
print(f"\nDisk Space in {working_dir}:")
print(f"  Total : {total_b / (1024**3):.2f} GB")
print(f"  Used  : {used_b / (1024**3):.2f} GB")
print(f"  Free  : {free_b / (1024**3):.2f} GB")
print("=" * 65)

### 3. Extract Dataset and Code to /kaggle/working/

Detects whether dataset was already extracted or is packaged in `.zip` files, and sets up writable directories.

In [ ]:
import os
import sys
import shutil
import zipfile
import gzip
import base64
import io
from pathlib import Path

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")
working_data = working_root / "data"
working_manifests = working_data / "manifests_seg"
working_data.mkdir(parents=True, exist_ok=True)
working_manifests.mkdir(parents=True, exist_ok=True)
working_ml = working_root / "ml" / "segmentation"

print("=" * 70)
print("1. UNPACKING AUTHORITATIVE SEGMENTATION SOURCE CODE:")
print("=" * 70)

EMBEDDED_CODE_B64 = (
    "..."
    "AAA="
)

print("Unpacking verified self-contained segmentation code suite...")
raw_code_zip = base64.b64decode(EMBEDDED_CODE_B64)
with zipfile.ZipFile(io.BytesIO(raw_code_zip), "r") as zf:
    zf.extractall(working_root)
print(f"[OK] Authoritative segmentation suite unpacked directly to {working_ml}.")

# Robustness Guard: Verify and enforce critical code invariants in train_seg.py
train_py_path = working_ml / "train_seg.py"
if train_py_path.exists():
    with open(train_py_path, "r", encoding="utf-8") as f:
        train_content = f.read()
    
    # Enforce val_loader batch_size=1 invariant
    if "batch_size=cur_batch_size" in train_content:
        train_content = train_content.replace(
            "batch_size=cur_batch_size,\n        shuffle=False,",
            "batch_size=1,\n        shuffle=False,"
        )
    
    # Enforce non-contiguous tensor safety (.reshape instead of .view)
    if ".view(" in train_content:
        train_content = train_content.replace(
            "p = probs.view(probs.size(0), -1)",
            "p = probs.reshape(probs.size(0), -1)"
        )
        train_content = train_content.replace(
            "t = targets.view(targets.size(0), -1)",
            "t = targets.reshape(targets.size(0), -1)"
        )
        train_content = train_content.replace(
            "p = preds.view(preds.size(0), -1)",
            "p = preds.reshape(preds.size(0), -1)"
        )
    
    with open(train_py_path, "w", encoding="utf-8") as f:
        f.write(train_content)
    
    import py_compile
    py_compile.compile(str(train_py_path), doraise=True)
    print("  [VERIFIED] train_seg.py compilation verified (0 syntax errors, batch_size=1, .reshape invariant).")

print(f"Segmentation code ready: {(working_ml / 'train_seg.py').exists()}")

print("\n" + "=" * 70)
print("2. WRITING STANDARDIZED MANIFESTS (CRACK500 & CFD):")
print("=" * 70)

crack500_manifest = working_manifests / "crack500.csv"
cfd_manifest = working_manifests / "cfd.csv"

# Embedded standardized manifests (gzip compressed + base64 encoded)
EMBEDDED_CRACK500_MANIFEST_GZ_B64 = (
    "..."
)

EMBEDDED_CFD_MANIFEST_GZ_B64 = (
    "H4sIAPF25GoC/33YQW5TQRAG4T0SN7ES9/RMz8wJskPcIDJggsGRIjn3FwR2KVS7p35Tq2/3X55PT+fHl9Prj8Pz6fbr39ft5Xp5"
    "/fjh6/dv95e3/7f74zHufr48Hd5Ob+/+Xh6vpy/n693nTw+H1/PtfdAQNA8SQXrQEXQPBoLhQSEoDyaC6cFCsDzYCLYGcXwfxNED"
    "SIdLB6TDpQPS4dIB6XDpgHS4dEA6XDogHS4dkA6XDkiHSzdIN5dukG4u3SDdXLpBurl0g3Rz6Qbp5tIN0s2lG6SbSzdIN5dukG4u"
    "nZBOl05Ip0snpNOlE9Lp0gnpdOmEdLp0QjpdOiGdLp2QTpdOSKdLd0h3l+6Q7i7dId1dukO6u3SHdHfpDunu0h3S3aU7pLtLd0h3"
    "l+6Q7i49ID1cekB6uPSA9HDpAenh0gPSw6UHpIdLD0gPlx6QHi49ID1cekB6uHRBuly6IF0uXZAuly5Il0sXpMulC9Ll0gXpcumC"
    "dLl0QbpcuiBdLj0hPV16Qnq69IT0dOkJ6enSE9LTpSekp0tPSE+XnpCeLj0hPV16Qnq69IL0cukF6eXSC9LLpRekl0svSC+XXpBe"
    "Lr0gvVx6QXq59IL0cukF6eXSG9LbpTekt0tvSG+X3pDeLr0hvV16Q3q79Ib0dukN6e3SG9LbpTekt0rH8b30n4sHgSA8aAiaB4kg"
    "PegIugcDwfCgEJQHE8H0YCFYHmwELo2NLHwjC2xk4RtZYCML38gCG1n4RhbYyMI3ssBGFr6RBTay8I0ssJGFb2SBjSz+s5H9Bptd"
    "4BKyFgAA"
)

# 1. Check if manifests were extracted from dataset zip or present in working_data / input
if not crack500_manifest.exists() or crack500_manifest.stat().st_size == 0:
    for search_base in [working_data, input_root]:
        if search_base.is_dir():
            matches = [p for p in search_base.rglob("crack500.csv") if p != crack500_manifest and p.is_file() and p.stat().st_size > 0]
            if matches:
                shutil.copyfile(matches[0], crack500_manifest)
                print(f"[OK] Found and copied CRACK500 manifest from {matches[0]} -> {crack500_manifest}")
                break

if not cfd_manifest.exists() or cfd_manifest.stat().st_size == 0:
    for search_base in [working_data, input_root]:
        if search_base.is_dir():
            matches = [p for p in search_base.rglob("cfd.csv") if p != cfd_manifest and p.is_file() and p.stat().st_size > 0]
            if matches:
                shutil.copyfile(matches[0], cfd_manifest)
                print(f"[OK] Found and copied CFD manifest from {matches[0]} -> {cfd_manifest}")
                break

# 2. If not found in dataset, unpack embedded gzip+base64 manifests
if not crack500_manifest.exists() or crack500_manifest.stat().st_size == 0:
    with open(crack500_manifest, "wb") as f:
        f.write(gzip.decompress(base64.b64decode(EMBEDDED_CRACK500_MANIFEST_GZ_B64)))
    print(f"[OK] Written embedded CRACK500 manifest to {crack500_manifest}")

if not cfd_manifest.exists() or cfd_manifest.stat().st_size == 0:
    with open(cfd_manifest, "wb") as f:
        f.write(gzip.decompress(base64.b64decode(EMBEDDED_CFD_MANIFEST_GZ_B64)))
    print(f"[OK] Written embedded CFD manifest to {cfd_manifest}")

print(f"[OK] Manifests verified: {crack500_manifest} and {cfd_manifest}")

print("=" * 70)
print("3. DETECTING DATASET IMAGES (CRACK500 & CFD):")
print("=" * 70)

crack500_dir = working_data / "crack500"
cfd_dir = working_data / "cfd"

def find_child_case_insensitive(parent, name_lower):
    if not parent.is_dir():
        return None
    try:
        for child in parent.iterdir():
            if child.is_dir() and child.name.lower() == name_lower.lower():
                return child
    except Exception:
        pass
    return None

def verify_seg_layout(folder):
    """Check if folder has images and masks directories (case-insensitive)."""
    if not folder.is_dir():
        return False
    img = find_child_case_insensitive(folder, "images")
    msk = find_child_case_insensitive(folder, "masks")
    return img is not None and msk is not None

def setup_dataset_symlink(src_folder, target_dir):
    """Ensure target_dir has lowercase images and masks pointing to src_folder."""
    if src_folder.resolve() == target_dir.resolve():
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    src_img = find_child_case_insensitive(src_folder, "images")
    src_msk = find_child_case_insensitive(src_folder, "masks")
    
    tgt_img = target_dir / "images"
    tgt_msk = target_dir / "masks"
    
    if src_img:
        if tgt_img.is_symlink():
            tgt_img.unlink()
        if not tgt_img.exists():
            try:
                os.symlink(src_img, tgt_img, target_is_directory=True)
            except Exception:
                shutil.copytree(src_img, tgt_img, dirs_exist_ok=True)
                
    if src_msk:
        if tgt_msk.is_symlink():
            tgt_msk.unlink()
        if not tgt_msk.exists():
            try:
                os.symlink(src_msk, tgt_msk, target_is_directory=True)
            except Exception:
                shutil.copytree(src_msk, tgt_msk, dirs_exist_ok=True)

# 1. Print input diagnostic
print("[DIAGNOSTIC] Checking /kaggle/input contents...")
all_input_files = []
if input_root.is_dir():
    for root, dirs, files in os.walk(input_root):
        rel = os.path.relpath(root, input_root)
        if files:
            all_input_files.extend([Path(root) / f for f in files])
        if rel == ".":
            print(f"  /kaggle/input: {len(dirs)} subdirectories, {len(files)} files")
            for d in dirs:
                print(f"    - {d}/")
else:
    print("  /kaggle/input directory not found!")

# 2. Extract ANY zip archive found in /kaggle/input (case-insensitive .zip / .ZIP)
if input_root.is_dir():
    for zp in input_root.rglob("*"):
        if zp.is_file() and zp.suffix.lower() == ".zip":
            if "ml_code" in zp.name.lower():
                continue
            try:
                with zipfile.ZipFile(zp, "r") as zf:
                    nl = [n.lower() for n in zf.namelist()]
                    if any("crack" in n or "images" in n or "cfd" in n for n in nl):
                        print(f"[EXTRACT] Extracting {zp.name} ({zp.stat().st_size / 1e6:.1f} MB) -> {working_data}...")
                        zf.extractall(working_data)
            except Exception as ze:
                print(f"Zip extraction note on {zp}: {ze}")

# 3. Locate CRACK500 (search across /kaggle/working and /kaggle/input)
found_crack500 = None

# A: Search for folder named crack500 (case-insensitive) with images and masks
for search_base in [working_data, input_root]:
    if not search_base.is_dir() or found_crack500:
        continue
    for p in search_base.rglob("*"):
        if p.is_dir() and "crack500" in p.name.lower() and verify_seg_layout(p):
            found_crack500 = p
            break

# B: Search for any folder that has images/masks containing CRACK500 sample 2016*.jpg
if not found_crack500:
    for search_base in [working_data, input_root]:
        if not search_base.is_dir() or found_crack500:
            continue
        c500_samples = list(search_base.rglob("*20160222*"))
        for s in c500_samples:
            p_cand = s.parent
            while p_cand != p_cand.parent and p_cand != search_base.parent:
                if verify_seg_layout(p_cand):
                    found_crack500 = p_cand
                    break
                p_cand = p_cand.parent
            if found_crack500:
                break

if found_crack500:
    print(f"[DETECT] Found CRACK500 at: {found_crack500}")
    setup_dataset_symlink(found_crack500, crack500_dir)

# 4. Locate CFD
found_cfd = None
for search_base in [working_data, input_root]:
    if not search_base.is_dir() or found_cfd:
        continue
    for p in search_base.rglob("*"):
        if p.is_dir() and "cfd" in p.name.lower() and verify_seg_layout(p):
            found_cfd = p
            break

if not found_cfd:
    for search_base in [working_data, input_root]:
        if not search_base.is_dir() or found_cfd:
            continue
        cfd_samples = list(search_base.rglob("*001.jpg*"))
        for s in cfd_samples:
            p_cand = s.parent
            while p_cand != p_cand.parent and p_cand != search_base.parent:
                if verify_seg_layout(p_cand):
                    found_cfd = p_cand
                    break
                p_cand = p_cand.parent
            if found_cfd:
                break

if found_cfd:
    print(f"[DETECT] Found CFD at: {found_cfd}")
    setup_dataset_symlink(found_cfd, cfd_dir)

# 5. Verification Guard
has_c500 = (crack500_dir / "images").exists() and (crack500_dir / "masks").exists()
if not has_c500:
    print("=" * 80)
    print("ACTION REQUIRED: KAGGLE DATASET INPUT NOT FOUND IN /kaggle/input")
    print("=" * 80)
    if not input_root.is_dir() or not list(input_root.iterdir()):
        print("  --> /kaggle/input is currently EMPTY (0 datasets attached).")
    else:
        print(f"  --> Items in /kaggle/input ({len(all_input_files)} total files found):")
        for f in all_input_files[:15]:
            print(f"      - {f}")
    print("-" * 80)
    print("HOW TO ATTACH YOUR DATASET TO THIS NOTEBOOK:")
    print("  1. Look at the RIGHT-HAND panel of your Kaggle notebook.")
    print("  2. In the 'Input' section, click '+ Add Input'.")
    print("  3. In the popup, click the 'Your Work' tab (do NOT click Competitions).")
    print("  4. Click the '+' button next to 'InfraScan Crack500 Seg'.")
    print("  5. Re-run this cell, and training will begin immediately!")
    print("=" * 80)
    raise FileNotFoundError("CRACK500 dataset not found in /kaggle/input! Please click '+ Add Input' in the right sidebar -> 'Your Work' tab -> '+' on 'InfraScan Crack500 Seg'.")


# 4. Verify manifests and sample image resolution
import pandas as pd
df_c500 = pd.read_csv(crack500_manifest)
df_cfd = pd.read_csv(cfd_manifest)
print(f"\n[VERIFIED] Crack500 Manifest: {len(df_c500):,} pairs across {df_c500['split'].value_counts().to_dict()}")
print(f"[VERIFIED] CFD Manifest     : {len(df_cfd):,} pairs across {df_cfd['split'].value_counts().to_dict()}")
sample_c500 = working_data / df_c500['image_path'].iloc[0]
assert sample_c500.exists(), f"Path resolution failed: {sample_c500}"
print(f"[OK] Verified sample image: {sample_c500}")
print("[OK] Dataset preparation complete! Ready for training.")


### 4. Setup Python Path and Import Segmentation Modules

Import the standardized segmentation package from `/kaggle/working`.

In [ ]:
import sys
import importlib
from pathlib import Path

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# Flush any cached modules to guarantee using freshly unpacked code
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("ml.") or mod_name == "ml":
        del sys.modules[mod_name]

from ml.segmentation import config, train_seg

print("[OK] Segmentation modules imported successfully (cache cleared).")
print(f"Encoder Backbone     : {config.ENCODER_BACKBONE}")
print(f"Training Crop Size   : {config.IMAGE_SIZE_TRAIN}x{config.IMAGE_SIZE_TRAIN}")
print(f"Evaluation Pad Mult  : {config.PAD_MULTIPLE_EVAL}")
print(f"Batch Size           : {config.BATCH_SIZE}")
print(f"Early Stop Patience  : {config.EARLY_STOPPING_PATIENCE}")


### 5. Sanity Debug Run (200 Train Images, 50 Val Images, 1 Epoch)

Verify pipeline forward-backward pass, loss computation, and metric logging.

In [ ]:
print("Starting Sanity Debug Run...")
debug_res = train_seg.train(
    model_name="unet",
    manifest_csv="/kaggle/working/data/manifests_seg/crack500.csv",
    project_dir="/kaggle/working/debug_segmentation",
    run_name="debug_run",
    data_root="/kaggle/working/data",
    debug=True,
    device_str="cuda:0",
)
print("[OK] Debug run completed successfully!")

### 6. Full Training Run 1: U-Net (ResNet-34, 40 Epochs, Seed 42)

Train baseline U-Net architecture with ImageNet pretrained encoder.
Outputs write to `/kaggle/working/models/segmentation/unet_crack500/`.

In [ ]:
models_dir = "/kaggle/working/models/segmentation"
manifest_file = "/kaggle/working/data/manifests_seg/crack500.csv"
data_root_dir = "/kaggle/working/data"

print("Starting Full Training of U-Net Baseline...")
unet_train_res = train_seg.train(
    model_name="unet",
    manifest_csv=manifest_file,
    project_dir=models_dir,
    run_name="unet_crack500",
    data_root=data_root_dir,
    debug=False,
    device_str="cuda:0",
)
print("[OK] U-Net training complete!")

### 7. Full Training Run 2: U-Net++ (ResNet-34, 40 Epochs, Seed 42)

Train comparison architecture U-Net++ with nested skip pathways under identical hyperparameters and seed.
Outputs write to `/kaggle/working/models/segmentation/unetplusplus_crack500/`.

In [ ]:
print("Starting Full Training of U-Net++ Comparison Model...")
unetplus_train_res = train_seg.train(
    model_name="unetplusplus",
    manifest_csv=manifest_file,
    project_dir=models_dir,
    run_name="unetplusplus_crack500",
    data_root=data_root_dir,
    debug=False,
    device_str="cuda:0",
)
print("[OK] U-Net++ training complete!")

### 8. Validation Comparison Table (Phase 1 Decision)

Compare both architectures on the **VALIDATION split** only to identify the superior architecture.

In [ ]:
import pandas as pd
import json
from pathlib import Path

val_records = []
for rname in ["unet_crack500", "unetplusplus_crack500"]:
    rdir = Path(models_dir) / rname
    rcard_path = rdir / "run_card.json"
    sweep_path = rdir / "threshold_sweep_val.csv"
    if rcard_path.exists():
        with open(rcard_path, "r") as f:
            card = json.load(f)
        b_val = card.get("best_validation_results", {})
        val_records.append({
            "Architecture": card.get("model_name"),
            "Run Name": rname,
            "Best Val Dice (th=0.5)": b_val.get("best_val_dice_at_05"),
            "Optimal Threshold": b_val.get("optimal_threshold"),
            "Optimal Val Dice": b_val.get("optimal_val_dice"),
            "Checkpoint": str(rdir / "best.pth"),
        })

val_df = pd.DataFrame(val_records)
print("=== VALIDATION SPLIT ARCHITECTURE COMPARISON ===")
print(val_df.to_markdown(index=False))

# Pick winning architecture based strictly on validation Dice
if len(val_df) > 1:
    winner = val_df.sort_values(by="Optimal Val Dice", ascending=False).iloc[0]
    print(f"\n[DECISION] Winner Architecture Selected on Validation Dice: {winner['Architecture']} (Run: {winner['Run Name']})")
    print(f"[DECISION] Optimal Probability Threshold Selected       : {winner['Optimal Threshold']:.2f}")

### 9. Package Model Checkpoints & Run Cards to /kaggle/working/segmentation_results.zip

Package all trained models, run cards, and validation sweep tables into a downloadable ZIP archive.

In [ ]:
import os
import shutil
from pathlib import Path

out_zip = "/kaggle/working/segmentation_results.zip"
models_p = Path("/kaggle/working/models")

if models_p.exists():
    print(f"Archiving {models_p} to {out_zip}...")
    shutil.make_archive(
        base_name="/kaggle/working/segmentation_results",
        format="zip",
        root_dir="/kaggle/working",
        base_dir="models",
    )
    size_mb = os.path.getsize(out_zip) / (1024**2)
    print(f"[OK] Created {out_zip} ({size_mb:.2f} MB).")
    print("    Download this file from the Kaggle Output sidebar after Save Version!")
else:
    print(f"[!] Directory not found: {models_p}")
